<a href="https://colab.research.google.com/github/DeeveshBeegun/machine_learning_handson/blob/main/spam_neural_network_hands_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-On: Building a Neural Network for Spam Detection

## Goal
In this hands-on, you will build the **architecture** of a neural network that could classify an email as **Spam** or **Not Spam**.

You will:
1. Convert emails into numerical features
2. Identify the input size
3. Choose hidden layers and neurons
4. Choose activation functions
5. Build the neural network using Keras
6. Inspect the architecture
7. Perform a forward pass


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display, Markdown
import ipywidgets as widgets

print("TensorFlow version:", tf.__version__)


## 1. Look at some example emails

Before building a neural network, inspect a few emails and think about what information could help distinguish spam from legitimate messages.


In [ ]:
emails = [
    "Congratulations! You have WON a FREE prize!!! Click http://claim-now.com",
    "Hi Sarah, can we move tomorrow's project meeting to 2 PM?",
    "URGENT!!! Your account has been selected for a CASH reward. Click now!",
    "Please find attached the report from yesterday's meeting.",
    "WIN MONEY NOW!!! FREE FREE FREE!!! Visit http://winner.com"
]

for i, email in enumerate(emails, start=1):
    print(f"\nEmail {i}")
    print("-" * 60)
    print(email)


### Discussion

Can a neural network directly use these sentences as numerical inputs?

For this introductory exercise, we will manually convert each email into a small set of numerical features.


In [ ]:
spam_words = [
    "free",
    "win",
    "won",
    "money",
    "cash",
    "prize",
    "urgent",
    "reward"
]

def extract_features(email):
    words = email.lower().split()

    spam_word_count = sum(
        word.strip("!.,?") in spam_words
        for word in words
    )

    exclamation_count = email.count("!")
    link_count = email.lower().count("http")
    email_length = len(email)

    uppercase_words = sum(
        word.isupper() and len(word) > 1
        for word in email.split()
    )

    return [
        spam_word_count,
        exclamation_count,
        link_count,
        email_length,
        uppercase_words
    ]


## 2. Convert the emails into neural-network inputs

We will represent every email using five numerical features:

- Number of suspicious/spam words
- Number of exclamation marks
- Number of links
- Email length
- Number of uppercase words


In [ ]:
features = np.array([
    extract_features(email)
    for email in emails
])

feature_names = [
    "Spam words",
    "Exclamation marks",
    "Links",
    "Email length",
    "Uppercase words"
]

df = pd.DataFrame(
    features,
    columns=feature_names
)

df["Email"] = emails

display(df)


## Interactive Exercise 1 — Identify the input size

How many numerical values enter the neural network for each email?


In [ ]:
input_answer = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description="Inputs:"
)

check_button = widgets.Button(
    description="Check answer"
)

output = widgets.Output()

def check_input_size(b):
    with output:
        output.clear_output()

        if input_answer.value == 5:
            print("✅ Correct!")
            print("We have 5 features, so the network receives 5 input values.")
        else:
            print("❌ Try again.")
            print("Count how many numerical features describe each email.")

check_button.on_click(check_input_size)

display(
    input_answer,
    check_button,
    output
)


## 3. Design the hidden layers

A possible architecture could look like this:

`5 inputs → 8 neurons → 4 neurons → 1 output`

But there is no need to use exactly these numbers.

Use the sliders below to choose your own hidden-layer sizes.


In [ ]:
hidden1_widget = widgets.IntSlider(
    value=8,
    min=1,
    max=32,
    step=1,
    description="Hidden 1:"
)

hidden2_widget = widgets.IntSlider(
    value=4,
    min=1,
    max=32,
    step=1,
    description="Hidden 2:"
)

display(
    hidden1_widget,
    hidden2_widget
)


## 4. Choose activation functions

For this binary-classification network:

- Hidden layers will use **ReLU**
- The output layer will use **Sigmoid**

The sigmoid output gives a value between 0 and 1, which we can interpret as a spam probability.


In [ ]:
hidden_activation = widgets.Dropdown(
    options=["Choose...", "relu", "sigmoid", "softmax"],
    description="Hidden:"
)

output_activation = widgets.Dropdown(
    options=["Choose...", "relu", "sigmoid", "softmax"],
    description="Output:"
)

activation_button = widgets.Button(
    description="Check choices"
)

activation_feedback = widgets.Output()

def check_activations(b):

    with activation_feedback:

        activation_feedback.clear_output()

        if (
            hidden_activation.value == "relu"
            and output_activation.value == "sigmoid"
        ):
            print("✅ Great!")
            print()
            print("Hidden layers → ReLU")
            print("Output layer → Sigmoid")
            print()
            print("Sigmoid gives us one value between 0 and 1.")

        else:
            print("❌ Try again.")
            print()
            print("Hint:")
            print("• Hidden layers usually use ReLU.")
            print("• The output should be one value between 0 and 1.")

activation_button.on_click(check_activations)

display(
    hidden_activation,
    output_activation,
    activation_button,
    activation_feedback
)


## 5. Build the neural network

Now create the network using TensorFlow/Keras.

Notice that we are only **building the architecture**. We are not compiling or training the model yet.


In [ ]:
def build_network(hidden1=8, hidden2=4):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(shape=(5,)),

        tf.keras.layers.Dense(
            hidden1,
            activation="relu",
            name="hidden_layer_1"
        ),

        tf.keras.layers.Dense(
            hidden2,
            activation="relu",
            name="hidden_layer_2"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="spam_probability"
        )

    ])

    return model


In [ ]:
model = build_network(
    hidden1=hidden1_widget.value,
    hidden2=hidden2_widget.value
)

model.summary()


### Think about the output layer

Why is there only **one output neuron**?

Because this is binary classification. The single sigmoid output can be interpreted as a probability:

- close to 0 → likely not spam
- close to 1 → likely spam


## 6. Visualize the network


In [ ]:
tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True
)


## 7. Interactive architecture experiment

Change the hidden-layer sizes and observe how the number of trainable parameters changes.


In [ ]:
@widgets.interact(
    hidden_layer_1=(1, 32, 1),
    hidden_layer_2=(1, 32, 1)
)
def experiment_with_network(
    hidden_layer_1=8,
    hidden_layer_2=4
):

    experiment_model = tf.keras.Sequential([

        tf.keras.layers.Input(shape=(5,)),

        tf.keras.layers.Dense(
            hidden_layer_1,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            hidden_layer_2,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )

    ])

    print()
    print(
        f"Architecture: 5 → {hidden_layer_1} → "
        f"{hidden_layer_2} → 1"
    )

    print()
    print(
        "Number of parameters:",
        experiment_model.count_params()
    )


### Challenge

Try these architectures:

- `5 → 4 → 2 → 1`
- `5 → 8 → 4 → 1`
- `5 → 32 → 16 → 1`

Which has the most parameters?

Does a larger network automatically mean a better network?


## 8. Inspect individual layers


In [ ]:
model = build_network(8, 4)

for i, layer in enumerate(model.layers):

    print(f"Layer {i + 1}")
    print("Name:", layer.name)
    print("Number of neurons:", layer.units)
    print("Activation:", layer.activation.__name__)
    print("-" * 40)


## 9. Quick architecture quiz


In [ ]:
questions = [
    {
        "question": "We have 5 email features. How many input values?",
        "options": ["1", "2", "5", "10"],
        "answer": "5"
    },
    {
        "question": "How many output neurons are needed for this binary classifier?",
        "options": ["1", "2", "5", "10"],
        "answer": "1"
    },
    {
        "question": "Which activation should we use in the final layer?",
        "options": ["ReLU", "Sigmoid", "None", "Linear"],
        "answer": "Sigmoid"
    }
]

for q in questions:

    print("\n" + q["question"])

    dropdown = widgets.Dropdown(
        options=q["options"]
    )

    button = widgets.Button(
        description="Check"
    )

    feedback = widgets.Output()

    def create_checker(question, widget, feedback_box):

        def check(b):

            with feedback_box:

                feedback_box.clear_output()

                if widget.value == question["answer"]:
                    print("✅ Correct!")
                else:
                    print("❌ Try again.")

        return check

    button.on_click(
        create_checker(q, dropdown, feedback)
    )

    display(
        dropdown,
        button,
        feedback
    )


## 10. Forward pass without training

We can send an email through the network even before it has learned anything.

However, because the network has not been trained, its prediction is not meaningful yet.

This step is only to demonstrate the flow:

`Email → Features → Neural Network → Output`


In [ ]:
model = build_network(8, 4)

sample_email = """
Congratulations!!!
You have WON a FREE cash prize.
Click http://claim-now.com
"""

sample_features = np.array([
    extract_features(sample_email)
])

print("Email:")
print(sample_email)

print("\nFeatures:")
print(sample_features)

prediction = model(sample_features)

print("\nNetwork output:")
print(prediction.numpy())

print("\n⚠️ The network has NOT been trained, so this output is not meaningful yet.")


## 11. Try your own email


In [ ]:
email_box = widgets.Textarea(
    value="Congratulations! You won a FREE prize!!!",
    placeholder="Write an email here...",
    description="Email:",
    layout=widgets.Layout(
        width="80%",
        height="120px"
    )
)

analyse_button = widgets.Button(
    description="Pass through network"
)

prediction_output = widgets.Output()

def analyse_email(b):

    with prediction_output:

        prediction_output.clear_output()

        email = email_box.value

        x = np.array([
            extract_features(email)
        ])

        print("Numerical features")
        print("------------------")

        for name, value in zip(
            feature_names,
            x[0]
        ):
            print(f"{name:20s}: {value}")

        result = model(x).numpy()[0][0]

        print()
        print("Network output:")
        print(round(float(result), 4))

        print()
        print("⚠️ Remember: the network has NOT been trained.")
        print("This only demonstrates information flowing through the architecture.")

analyse_button.on_click(analyse_email)

display(
    email_box,
    analyse_button,
    prediction_output
)


# Final Group Challenge

Design your own spam-detection neural network.

Decide:

1. Number of input values
2. Number of hidden layers
3. Number of neurons in each hidden layer
4. Activation function for hidden layers
5. Number of output neurons
6. Activation function for the output

Then create your network using:

```python
model = tf.keras.Sequential([
    ...
])
```

Finally run:

```python
model.summary()
```

Be ready to explain:

- Why did you choose that number of neurons?
- Why is the final layer only one neuron?
- Why is sigmoid useful for the output?
- How many parameters does your network contain?
- What changes when you make the network larger?

## Important rule

Do **not** use:

```python
model.compile()
model.fit()
```

Today, you are only building the neural network.

Training, loss functions, gradient descent, and backpropagation come in the next hands-on.
